In [1]:
# ================================
# GTZAN COMPLETE PIPELINE
# Preprocess 3-sec + 30-sec
# Train models
# Compare results
# ================================

import re
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score


In [2]:
# ================================
# GLOBAL SETTINGS
# ================================

TEST_SIZE = 0.2
RANDOM_STATE = 42


# ================================
# PREPROCESSING FUNCTION
# ================================

def preprocess_gtzan_csv(INPUT_CSV: str, OUT_PREFIX: str,
                         label_col="label",
                         fname_col="filename"):

    print(f"\n=== Preprocessing {INPUT_CSV} ===")

    df = pd.read_csv(INPUT_CSV)

    def _make_song_id_from_filename(fname: str) -> str:
        base = fname.split("/")[-1]
        base = re.sub(r"\.wav$", "", base)
        return base

    df["song_id"] = df[fname_col].astype(str).apply(_make_song_id_from_filename)

    # Separate features and labels
    drop_cols = [label_col, fname_col, "song_id"]
    y_raw = df[label_col].astype(str).values
    song_ids = df["song_id"].astype(str).values

    X = df.drop(columns=drop_cols, errors="ignore")
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.dropna(axis=1, how="all")

    # Encode labels
    le = LabelEncoder()
    y = le.fit_transform(y_raw)

    # Leakage-safe split by song_id
    unique_songs = pd.unique(song_ids)
    song_to_label = {}
    for sid, lab in zip(song_ids, y):
        song_to_label.setdefault(sid, lab)
    song_labels = np.array([song_to_label[s] for s in unique_songs])

    train_songs, test_songs = train_test_split(
        unique_songs,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=song_labels
    )

    train_mask = np.isin(song_ids, train_songs)
    test_mask  = np.isin(song_ids, test_songs)

    X_train = X.values[train_mask]
    X_test  = X.values[test_mask]
    y_train = y[train_mask]
    y_test  = y[test_mask]
    song_train = song_ids[train_mask]
    song_test  = song_ids[test_mask]

    # Impute missing values
    imputer = SimpleImputer(strategy="median")
    X_train_imp = imputer.fit_transform(X_train)
    X_test_imp  = imputer.transform(X_test)

    # Standardize
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_test_scaled  = scaler.transform(X_test_imp)

    # Save arrays
    np.save(f"{OUT_PREFIX}_X_train.npy", X_train_scaled)
    np.save(f"{OUT_PREFIX}_X_test.npy", X_test_scaled)
    np.save(f"{OUT_PREFIX}_y_train.npy", y_train)
    np.save(f"{OUT_PREFIX}_y_test.npy", y_test)
    np.save(f"{OUT_PREFIX}_song_test.npy", song_test)

    print(f"[{OUT_PREFIX}] Saved arrays.")
    print("Train label counts:", np.bincount(y_train))
    print("Test label counts: ", np.bincount(y_test))




In [3]:
# ================================
# MODEL TRAINING + EVALUATION
# ================================

def load_split(prefix: str):
    X_train = np.load(f"{prefix}_X_train.npy")
    X_test  = np.load(f"{prefix}_X_test.npy")
    y_train = np.load(f"{prefix}_y_train.npy")
    y_test  = np.load(f"{prefix}_y_test.npy")
    song_test = np.load(f"{prefix}_song_test.npy", allow_pickle=True)
    return X_train, X_test, y_train, y_test, song_test


def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
    }


def majority_vote(y_pred, y_true, song_ids):
    dfp = pd.DataFrame({"song": song_ids, "pred": y_pred, "true": y_true})
    y_song_pred = dfp.groupby("song")["pred"].agg(lambda x: x.mode().iloc[0])
    y_song_true = dfp.groupby("song")["true"].first().reindex(y_song_pred.index)
    return y_song_true.values, y_song_pred.values


def get_models():
    return {
        "LogReg": LogisticRegression(max_iter=4000, multi_class="multinomial"),
        "SVM_RBF": SVC(kernel="rbf", C=10, gamma="scale"),
        "RandomForest": RandomForestClassifier(n_estimators=400, random_state=42),
        "GradBoost": GradientBoostingClassifier(random_state=42),
    }


def run_experiment(prefix: str, dataset_name: str, song_level=False):
    X_train, X_test, y_train, y_test, song_test = load_split(prefix)
    rows = []

    for name, model in get_models().items():

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Row-level
        m = compute_metrics(y_test, y_pred)
        rows.append({
            "dataset": dataset_name,
            "evaluation": "row_level",
            "model": name,
            **m
        })

        # Song-level (for 3-sec)
        if song_level:
            y_song_true, y_song_pred = majority_vote(y_pred, y_test, song_test)
            m_song = compute_metrics(y_song_true, y_song_pred)
            rows.append({
                "dataset": dataset_name,
                "evaluation": "song_majority_vote",
                "model": name,
                **m_song
            })

    return pd.DataFrame(rows)

In [4]:
# ================================
# MAIN EXECUTION
# ================================

if __name__ == "__main__":

    # 1 Preprocess both datasets
    preprocess_gtzan_csv("features_3_sec.csv", "gtzan_3sec")
    preprocess_gtzan_csv("features_30_sec.csv", "gtzan_30sec")

    # 2 Train + compare
    df_3  = run_experiment("gtzan_3sec",  "3_sec_features",  song_level=True)
    df_30 = run_experiment("gtzan_30sec", "30_sec_features", song_level=False)

    results = pd.concat([df_3, df_30], ignore_index=True)

    results_sorted = results.sort_values(
        by=["dataset", "evaluation", "accuracy", "macro_f1"],
        ascending=[True, True, False, False]
    ).reset_index(drop=True)

    print("\n=== FINAL COMPARISON RESULTS ===")
    print(results_sorted)

    results_sorted.to_csv("gtzan_model_comparison.csv", index=False)
    print("\nSaved: gtzan_model_comparison.csv")


=== Preprocessing features_3_sec.csv ===
[gtzan_3sec] Saved arrays.
Train label counts: [800 799 798 799 798 800 800 800 800 798]
Test label counts:  [200 199 199 200 200 200 200 200 200 200]

=== Preprocessing features_30_sec.csv ===
[gtzan_30sec] Saved arrays.
Train label counts: [80 80 80 80 80 80 80 80 80 80]
Test label counts:  [20 20 20 20 20 20 20 20 20 20]


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



=== FINAL COMPARISON RESULTS ===
            dataset          evaluation         model  accuracy  macro_f1  \
0   30_sec_features           row_level  RandomForest  0.805000  0.802921   
1   30_sec_features           row_level     GradBoost  0.780000  0.780255   
2   30_sec_features           row_level       SVM_RBF  0.765000  0.764319   
3   30_sec_features           row_level        LogReg  0.740000  0.740122   
4    3_sec_features           row_level       SVM_RBF  0.913413  0.913266   
5    3_sec_features           row_level  RandomForest  0.875375  0.874992   
6    3_sec_features           row_level     GradBoost  0.812813  0.812974   
7    3_sec_features           row_level        LogReg  0.719720  0.717884   
8    3_sec_features  song_majority_vote       SVM_RBF  0.913413  0.913266   
9    3_sec_features  song_majority_vote  RandomForest  0.875375  0.874992   
10   3_sec_features  song_majority_vote     GradBoost  0.812813  0.812974   
11   3_sec_features  song_majority_vote   